# ScreenSense — Behavioural Feature Engineering

This notebook tests whether engagement behaviours provide meaningful context beyond screen time alone.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/processed/screensense_clean.csv")

print(df.shape)

(10000, 21)


In [3]:
behaviour_cols = [
    "daily_screen_time_minutes",
    "session_duration_minutes",
    "sessions_per_day",
    "app_opens_per_day",
    "notifications_received_per_day"
]

thresholds = df[behaviour_cols].quantile(0.75)

print("75th Percentile Thresholds:")
display(thresholds)

75th Percentile Thresholds:


daily_screen_time_minutes         112.7
session_duration_minutes           48.7
sessions_per_day                    8.0
app_opens_per_day                  10.0
notifications_received_per_day     28.0
Name: 0.75, dtype: float64

In [4]:
df["high_screen_time"] = (
    df["daily_screen_time_minutes"] > thresholds["daily_screen_time_minutes"]
).astype(int)

df["long_sessions"] = (
    df["session_duration_minutes"] > thresholds["session_duration_minutes"]
).astype(int)

df["high_session_frequency"] = (
    df["sessions_per_day"] > thresholds["sessions_per_day"]
).astype(int)

df["high_app_opens"] = (
    df["app_opens_per_day"] > thresholds["app_opens_per_day"]
).astype(int)

df["high_notifications"] = (
    df["notifications_received_per_day"] > thresholds["notifications_received_per_day"]
).astype(int)

flag_cols = [
    "high_screen_time",
    "long_sessions",
    "high_session_frequency",
    "high_app_opens",
    "high_notifications"
]

display(df[flag_cols].mean().mul(100).round(2))

high_screen_time          24.98
long_sessions             24.94
high_session_frequency    15.45
high_app_opens            18.25
high_notifications        23.92
dtype: float64

In [5]:
behaviour_profiles = (
    df.groupby(flag_cols)
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

behaviour_profiles["record_pct"] = (
    behaviour_profiles["record_count"] / len(df) * 100
).round(2)

display(behaviour_profiles.head(15))

,high_screen_time,long_sessions,high_session_frequency,high_app_opens,high_notifications,record_count,record_pct
0,0,0,0,0,0,3158,31.58
1,0,0,0,0,1,985,9.85
16,1,0,0,0,0,819,8.19
8,0,1,0,0,0,797,7.97
2,0,0,0,1,0,706,7.06
4,0,0,1,0,0,543,5.43
24,1,1,0,0,0,496,4.96
9,0,1,0,0,1,255,2.55
17,1,0,0,0,1,247,2.47
3,0,0,0,1,1,221,2.21


### Observation

Distinct behavioural combinations exist across usage records. High screen time, long sessions, frequent app opens, and high session frequency often occur independently, supporting a multidimensional view of smartphone engagement.

In [6]:
def assign_persona(row):
    if row["high_screen_time"] == 0 and row[flag_cols].sum() == 0:
        return "Low-Intensity Users"

    elif row["high_screen_time"] == 1 and row["long_sessions"] == 1:
        return "Deep Engagement Users"

    elif row["high_app_opens"] == 1 or row["high_session_frequency"] == 1:
        return "Frequent Checkers"

    elif row["high_notifications"] == 1:
        return "Notification-Exposed Users"

    elif row["high_screen_time"] == 1:
        return "High Screen-Time Users"

    elif row["long_sessions"] == 1:
        return "Long-Session Users"

    else:
        return "Moderate Engagement Users"


df["behavioural_persona"] = df.apply(assign_persona, axis=1)

persona_distribution = (
    df["behavioural_persona"]
    .value_counts()
    .to_frame("record_count")
)

persona_distribution["record_pct"] = (
    persona_distribution["record_count"] / len(df) * 100
).round(2)

display(persona_distribution)

,record_count,record_pct
behavioural_persona,,
Low-Intensity Users,3158,31.58
Frequent Checkers,2780,27.80
Notification-Exposed Users,1487,14.87
Deep Engagement Users,959,9.59
High Screen-Time Users,819,8.19
Long-Session Users,797,7.97


In [7]:
persona_profile = (
    df.groupby("behavioural_persona")
    [
        [
            "daily_screen_time_minutes",
            "session_duration_minutes",
            "sessions_per_day",
            "app_opens_per_day",
            "notifications_received_per_day"
        ]
    ]
    .mean()
    .round(2)
)

display(persona_profile)

,daily_screen_time_minutes,session_duration_minutes,sessions_per_day,app_opens_per_day,notifications_received_per_day
behavioural_persona,,,,,
Deep Engagement Users,185.15,71.25,6.03,7.99,25.12
Frequent Checkers,74.45,32.03,7.67,10.20,25.08
High Screen-Time Users,180.88,27.42,5.31,6.96,23.07
Long-Session Users,66.89,67.50,5.27,7.07,23.02
Low-Intensity Users,49.62,24.11,5.27,7.09,22.78
Notification-Exposed Users,75.54,31.65,5.34,7.06,31.66


### Observation

The persona profiles show clearly different engagement patterns. High screen time, long sessions, frequent checking, and notification exposure represent distinct behavioural dimensions rather than a single usage pattern.

In [8]:
persona_wellbeing = (
    df.groupby("behavioural_persona")
    .agg(
        concern_rate=(
            "screen_time_concern",
            lambda x: (x == "Yes").mean() * 100
        ),
        severe_sleep_disruption_rate=(
            "sleep_disruption_from_phone",
            lambda x: (x == "Severe").mean() * 100
        ),
        negative_mental_health_rate=(
            "mental_health_impact",
            lambda x: x.isin(["Negative", "Very Negative"]).mean() * 100
        ),
        wellbeing_feature_usage_rate=(
            "digital_wellbeing_feature_used",
            lambda x: (x == "Yes").mean() * 100
        )
    )
    .round(2)
)

display(persona_wellbeing)

,concern_rate,severe_sleep_disruption_rate,negative_mental_health_rate,wellbeing_feature_usage_rate
behavioural_persona,,,,
Deep Engagement Users,38.58,13.24,36.39,34.52
Frequent Checkers,37.99,13.99,34.32,34.42
High Screen-Time Users,39.56,15.63,35.90,36.02
Long-Session Users,36.26,11.17,34.50,35.38
Low-Intensity Users,36.67,13.65,36.13,36.89
Notification-Exposed Users,37.39,11.30,34.84,34.63


### Observation

Behavioural personas differ clearly in engagement patterns but show limited variation in self-reported wellbeing indicators. High Screen-Time records show the highest concern and severe sleep-disruption rates, though differences remain modest.

In [9]:
persona_category = pd.crosstab(
    df["behavioural_persona"],
    df["app_category"],
    normalize="index"
).mul(100).round(2)

display(persona_category)

app_category,Communication/Messaging,Education,Entertainment/Streaming,Finance/Banking,Food Delivery,Gaming,Health & Fitness,News & Media,Productivity,Shopping,Social Media,Travel
behavioural_persona,,,,,,,,,,,,
Deep Engagement Users,0.00,1.04,41.92,0.00,0.00,29.82,0.00,0.00,2.82,0.00,24.40,0.00
Frequent Checkers,7.73,9.64,10.14,6.73,1.37,10.86,8.56,6.01,9.68,7.84,18.88,2.55
High Screen-Time Users,5.37,1.95,10.74,0.00,0.00,15.38,0.12,0.24,3.30,0.24,62.64,0.00
Long-Session Users,0.00,10.79,30.99,0.00,0.00,29.61,0.00,0.00,15.56,0.00,13.05,0.00
Low-Intensity Users,8.49,10.10,3.64,9.02,1.20,7.19,14.34,8.39,11.27,12.13,10.96,3.26
Notification-Exposed Users,5.99,8.14,10.29,7.87,1.14,12.78,9.35,6.32,10.56,7.94,18.16,1.48


### Observation

Behavioural personas show distinct app-category patterns. High Screen-Time records are strongly concentrated in Social Media, while Deep Engagement and Long-Session profiles are dominated by Streaming and Gaming.

In [10]:
output_path = "../data/processed/screensense_engineered.csv"

df.to_csv(output_path, index=False)

print("Engineered dataset saved.")
print("Final Shape:", df.shape)
print(df["behavioural_persona"].value_counts())

Engineered dataset saved.
Final Shape: (10000, 27)
behavioural_persona
Low-Intensity Users           3158
Frequent Checkers             2780
Notification-Exposed Users    1487
Deep Engagement Users          959
High Screen-Time Users         819
Long-Session Users             797
Name: count, dtype: int64
